In [1]:
import torch
import torch.nn as nn

class HuberMapeFeatureLoss(nn.HuberLoss):
    def __init__(self, delta=1.0, reduction='mean', eps=1e-8):
        # Initialize the parent Huber Loss with 'none' to keep element-wise values
        super().__init__(reduction='none', delta=delta)
        self.user_reduction = reduction
        self.eps = eps # Safeguard against dividing by zero sales

    def forward(self, input: torch.Tensor, target: torch.Tensor, x_feature: torch.Tensor) -> torch.Tensor:
        """
        input: Model predictions (y_hat)
        target: Actual values (y)
        x_feature: Binary tensor (0 or 1) of the same shape as input/target
        """
        # 1. Calculate standard Huber Loss element-wise
        huber_base = super().forward(input, target)
        
        # 2. Calculate Absolute Percentage Error element-wise
        # Formula: |target - input| / max(|target|, eps)
        absolute_error = torch.abs(target - input)
        absolute_percentage_error = absolute_error / torch.clamp(torch.abs(target), min=self.eps)
        
        # 3. Multiply the percentage error by your binary feature x
        # If x is 0, this term becomes 0. If x is 1, it retains the error.
        mape_penalty = x_feature * absolute_percentage_error
        
        # 4. Add the penalty to the base Huber Loss
        # Total Loss = HuberLoss + (x * APE)
        total_loss = huber_base + mape_penalty
        
        # 5. Apply the user's requested reduction (mean or sum)
        if self.user_reduction == 'mean':
            return total_loss.mean()
        elif self.user_reduction == 'sum':
            return total_loss.sum()
        else:
            return total_loss

In [2]:
import numpy as np
import pandas as pd
import torch

# 1. Set a random seed for reproducibility
np.random.seed(42)
num_samples = 5000

# 2. Generate the 4 continuous independent variables
x1 = np.random.uniform(1, 10, num_samples)     # Range 1 to 10
x2 = np.random.uniform(10, 50, num_samples)    # Range 10 to 50
x3 = np.random.uniform(0.1, 2.0, num_samples)  # Range 0.1 to 2.0 (kept small due to 400x multiplier)
x4 = np.random.uniform(5, 25, num_samples)     # Range 5 to 25

# 3. Generate x5 as a binary categorical variable (0 or 1)
# This is independent of the base equation but will be fed into the loss function
x5 = np.random.randint(0, 2, num_samples)

# 4. Generate the random error/noise term
error = np.random.normal(0, 15, num_samples)   # Mean=0, StdDev=15

# 5. Calculate Y based strictly on your linear equation
y = (1.2 * x1) + (4.3 * x2) + (400 * x3) + (5 * x4) + error

# 6. Combine everything into a clean Pandas DataFrame
dataset = pd.DataFrame({
    'x1': x1,
    'x2': x2,
    'x3': x3,
    'x4': x4,
    'x5_binary': x5,  # Our categorical loss-modulator feature
    'y_actual': y
})

# Quick peek at the generated data
print("--- First 5 Rows of the Generated Dataset ---")
print(dataset.head())

--- First 5 Rows of the Generated Dataset ---
         x1         x2        x3         x4  x5_binary    y_actual
0  4.370861  25.745421  0.809918  14.993405          1  500.852858
1  9.556429  28.937426  0.732533  19.934935          1  525.772536
2  7.587945  44.181896  0.434692  16.253336          1  461.625585
3  6.387926  23.600175  1.253807   6.666052          0  639.117875
4  2.404168  44.785987  1.005586   8.711605          0  660.891974


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ==========================================
# 1. DEFINE CUSTOM LOSS FUNCTION
# ==========================================
class HuberMapeFeatureLoss(nn.HuberLoss):
    def __init__(self, delta=1.0, reduction='mean', eps=1e-8):
        super().__init__(reduction='none', delta=delta)
        self.user_reduction = reduction
        self.eps = eps

    def forward(self, input: torch.Tensor, target: torch.Tensor, x_feature: torch.Tensor) -> torch.Tensor:
        # Step A: Native Huber Loss
        huber_base = super().forward(input, target)
        
        # Step B: Guarded absolute percentage error calculation
        absolute_error = torch.abs(target - input)
        absolute_percentage_error = absolute_error / torch.clamp(torch.abs(target), min=self.eps)
        
        # Step C: Modulate MAPE with the binary x feature
        mape_penalty = x_feature * absolute_percentage_error
        
        # Step D: Total Loss Summation
        total_loss = huber_base + mape_penalty
        
        if self.user_reduction == 'mean':
            return total_loss.mean()
        elif self.user_reduction == 'sum':
            return total_loss.sum()
        else:
            return total_loss

# ==========================================
# 2. GENERATE AND PREPARE DUMMY DATA
# ==========================================
np.random.seed(42)
num_samples = 5000

x1 = np.random.uniform(1, 10, num_samples)
x2 = np.random.uniform(10, 50, num_samples)
x3 = np.random.uniform(0.1, 2.0, num_samples)
x4 = np.random.uniform(5, 25, num_samples)
x5 = np.random.randint(0, 2, num_samples) # Binary feature for loss function
error = np.random.normal(0, 15, num_samples)

y = (1.2 * x1) + (4.3 * x2) + (400 * x3) + (5 * x4) + error

dataset = pd.DataFrame({'x1': x1, 'x2': x2, 'x3': x3, 'x4': x4, 'x5_binary': x5, 'y_actual': y})

# Convert to PyTorch Tensors
X_train = torch.tensor(dataset[['x1', 'x2', 'x3', 'x4']].values, dtype=torch.float32)
y_train = torch.tensor(dataset['y_actual'].values, dtype=torch.float32).unsqueeze(1)
x5_loss_feature = torch.tensor(dataset['x5_binary'].values, dtype=torch.float32).unsqueeze(1)

# ==========================================
# 3. INITIALIZE MODEL & CRITERION
# ==========================================
# Simple Linear Regression model (4 inputs mapping to 1 output)
model = nn.Linear(in_features=4, out_features=1)

# Initialize our custom loss
criterion = HuberMapeFeatureLoss(delta=1.0, reduction='mean')

# ==========================================
# 4. RUN TEST PASSES
# ==========================================
print("--- TEST STEP 1: FORWARD PASS ---")
# Generate baseline predictions using untrained model weights
predictions = model(X_train)
print(f"Predictions shape: {predictions.shape}")

# Calculate loss value
loss = criterion(predictions, y_train, x5_loss_feature)
print(f"Calculated Custom Loss Value: {loss.item():.4f}\n")

print("--- TEST STEP 2: BACKWARD PASS (GRADIENT FLOW) ---")
# Trigger the autograd engine to compute gradients
loss.backward()

# Display calculated gradients for the model parameters
print("Gradients successfully derived for model weights (x1, x2, x3, x4):")
print(model.weight.grad)
print("\nGradient successfully derived for model bias:")
print(model.bias.grad)

--- TEST STEP 1: FORWARD PASS ---
Predictions shape: torch.Size([5000, 1])
Calculated Custom Loss Value: 639.4373

--- TEST STEP 2: BACKWARD PASS (GRADIENT FLOW) ---
Gradients successfully derived for model weights (x1, x2, x3, x4):
tensor([[ -5.4765, -29.6859,  -1.0534, -15.1673]])

Gradient successfully derived for model bias:
tensor([-1.0009])


In [4]:
dataset.head()

,x1,x2,x3,x4,x5_binary,y_actual
0,4.370861,25.745421,0.809918,14.993405,1,500.852858
1,9.556429,28.937426,0.732533,19.934935,1,525.772536
2,7.587945,44.181896,0.434692,16.253336,1,461.625585
3,6.387926,23.600175,1.253807,6.666052,0,639.117875
4,2.404168,44.785987,1.005586,8.711605,0,660.891974


In [6]:
import torch
import torch.nn as nn

class DartsHuberMapeLoss(nn.HuberLoss):
    def __init__(self, delta=1.0, reduction='mean', eps=1e-8):
        super().__init__(reduction='none', delta=delta)
        self.user_reduction = reduction
        self.eps = eps

    def forward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # Darts PyTorch tensors are shaped: (batch_size, output_chunk_length, target_components)
        
        # 1. Unpack Ground Truths from the target tensor
        y_true = target[:, :, 0:1]      # Component 0: 'y_actual'
        x_feature = target[:, :, 1:2]   # Component 1: 'binary_x'
        
        # 2. Unpack the Prediction for 'y_actual'
        y_hat = input[:, :, 0:1]        # Component 0 prediction
        
        # 3. Calculate Huber Loss
        huber_base = super().forward(y_hat, y_true)
        
        # 4. Calculate Absolute Percentage Error (MAPE)
        absolute_error = torch.abs(y_true - y_hat)
        absolute_percentage_error = absolute_error / torch.clamp(torch.abs(y_true), min=self.eps)
        
        # 5. Apply the conditional binary penalty
        mape_penalty = x_feature * absolute_percentage_error
        total_loss = huber_base + mape_penalty
        
        # 6. Neutralize the model's dummy prediction for the second channel
        # This keeps the computational graph intact without affecting weights
        dummy_loss = 0.0 * input[:, :, 1:2].sum()
        total_loss = total_loss + dummy_loss
        
        if self.user_reduction == 'mean':
            return total_loss.mean()
        elif self.user_reduction == 'sum':
            return total_loss.sum()
        else:
            return total_loss

In [8]:
from dateutil.relativedelta import relativedelta
import datetime

In [19]:
d1 = datetime.date(2023,4,1)
d2 = datetime.date(2026,12,31)

In [20]:
(relativedelta(d2,d1).days) + (relativedelta(d2,d1).months*31) + (relativedelta(d2,d1).years*(365))

1373

In [21]:
(d2-d1).days

1370